# Evaluation Loop

Runs every active model on every active feature track, target, horizon and pseudo-out-of-sample fold, then saves per-run metrics, predictions and runtimes to `results/raw/<model>/<track>.parquet`.

This notebook only holds configuration. The logic lives in `src/`:

- `src/evaluation/` — folds, track preparation, metrics, the loop itself
- `src/models/` — model functions and `MODEL_REGISTRY`
- `src/features/quantum/` — quantum / classical reservoir feature generators (precomputed into tracks by `scripts/build_quantum_tracks.py`)

To add a model, see the docstring in `src/models/__init__.py`. To add a track, add an entry to `TRACK_CONFIG` below.

## 1. Environment setup

In [1]:
import os
import sys
from pathlib import Path

COLAB = False  # set to True when running in Google Colab

if COLAB:
    REPO_URL  = "https://github.com/manuelarceaguirre/capstone-quantum.git"
    BRANCH    = "clean"
    REPO_ROOT = Path("/content/capstone-quantum")

    %cd /content
    !rm -rf capstone-quantum
    !git clone -b {BRANCH} {REPO_URL}
    %cd /content/capstone-quantum
    !pip install -q pmdarima
    sys.path.insert(0, str(REPO_ROOT))
else:
    # local: walk up from the working directory to the repo root;
    # src/ is importable via `pip install -e .` (see environment.yml)
    REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

print("Repo root:", REPO_ROOT)

Repo root: /content/capstone-quantum


In [2]:
import warnings

import pandas as pd

from src.evaluation import (
    disaggregate_by_usrec,
    explode_predictions,
    generate_folds,
    load_results,
    load_tracks,
    run_evaluation,
    shared_track,
    summarize,
    write_csv,
)
from src.models import MODEL_REGISTRY

warnings.filterwarnings("ignore")

## 2. Configuration

In [3]:
# --- Paths ---
DATA_DIR    = REPO_ROOT / "data" / "processed"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# --- Window parameters ---
WINDOW_MONTHS = 120   # training window length
TEST_MONTHS   = 1     # test window length
STEP_MONTHS   = 1     # fold step size
MAX_FOLDS     = None  # set to a small int for a quick smoke test
RUN_BY        = "Manuel"  # your name, recorded on every result row (default: OS user)
RESUME        = False # True: skip runs already saved in results/ (e.g. after a crash or
                      # Colab disconnect). Keep False after changing a model's code.

# --- What to run ---
HORIZON_LIST  = [1, 3, 6]
TARGET_LIST   = ["INDPRO", "PAYEMS", "CPIAUCSL", "S&P 500"]
QUANTUM_TRACKS = ["Q_su_pce", "Q_su_pauli", "Q_su_bitstring", "Q_su_single_bit", "Q_heisenberg_pce"]
ACTIVE_TRACKS = ['B']                         # any keys of TRACK_CONFIG
ACTIVE_MODELS = ['adaboost']       # any keys of MODEL_REGISTRY

# --- Track registry: TRACK_CONFIG[track][target] = {"type", "path"} ---
TRACK_CONFIG = {
    "A":      shared_track("dataframe", DATA_DIR / "track_A_full.parquet", TARGET_LIST),
    "B":      {t: {"type": "dataframe", "path": DATA_DIR / f"track_b_{t}.parquet"} for t in TARGET_LIST},
    "C":      shared_track("pipeline", DATA_DIR / "track_C_pipe.pkl", TARGET_LIST),       # LASSO, tabled
    "D":      shared_track("pipeline", DATA_DIR / "track_D_pipe.pkl", TARGET_LIST),       # PCA ~30 factors
    "D_mini": shared_track("pipeline", DATA_DIR / "track_D_mini_pipe.pkl", TARGET_LIST),  # PCA 15 factors
    # reservoir tracks: built by scripts/build_quantum_tracks.py
    "QRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_QRC_{t}.parquet"} for t in TARGET_LIST},
    "CRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_CRC_{t}.parquet"} for t in TARGET_LIST},
    # quantum feature tracks: built by notebooks/quantum_data_prep.ipynb
    **{q: {t: {"type": "dataframe", "path": DATA_DIR / f"track_{q}_{t}.parquet"} for t in TARGET_LIST}
       for q in QUANTUM_TRACKS},
}

MODELS = {name: MODEL_REGISTRY[name] for name in ACTIVE_MODELS}

print(f"Window:   {WINDOW_MONTHS}m train / {TEST_MONTHS}m test / {STEP_MONTHS}m step")
print(f"Targets:  {TARGET_LIST}")
print(f"Horizons: {HORIZON_LIST}")
print(f"Tracks:   {ACTIVE_TRACKS}")
print(f"Models:   {ACTIVE_MODELS}")

Window:   120m train / 1m test / 1m step
Targets:  ['INDPRO', 'PAYEMS', 'CPIAUCSL', 'S&P 500']
Horizons: [1, 3, 6]
Tracks:   ['B']
Models:   ['adaboost']


## 3. Load data and build folds

In [4]:
track_data       = load_tracks(TRACK_CONFIG, ACTIVE_TRACKS, TARGET_LIST)
stationary_panel = pd.read_parquet(DATA_DIR / "stationary_panel.parquet")
targets          = pd.read_parquet(DATA_DIR / "targets.parquet")
usrec            = pd.read_parquet(DATA_DIR / "usrec.parquet")

print(f"Stationary panel {stationary_panel.shape}  {stationary_panel.index.min().date()} -> {stationary_panel.index.max().date()}")
print(f"Targets          {targets.shape}")

  Track B        target INDPRO     loaded   shape=(767, 10)
  Track B        target PAYEMS     loaded   shape=(767, 10)
  Track B        target CPIAUCSL   loaded   shape=(767, 12)
  Track B        target S&P 500    loaded   shape=(767, 12)
Stationary panel (767, 123)  1962-04-01 -> 2026-02-01
Targets          (806, 20)


In [5]:
# all feature panels share the stationary panel's monthly index
folds = generate_folds(stationary_panel.index, WINDOW_MONTHS, TEST_MONTHS, STEP_MONTHS)
if MAX_FOLDS is not None:
    folds = folds[:MAX_FOLDS]

print(f"Total folds: {len(folds)}")
print(f"  first: train {folds[0]['train_start'].date()} -> {folds[0]['train_end'].date()}, test {folds[0]['test_start'].date()}")
print(f"  last:  train {folds[-1]['train_start'].date()} -> {folds[-1]['train_end'].date()}, test {folds[-1]['test_start'].date()}")

Total folds: 647
  first: train 1962-04-01 -> 1972-03-01, test 1972-04-01
  last:  train 2016-02-01 -> 2026-01-01, test 2026-02-01


## 4. Run

Each track's results are saved to `results/raw/<model>/<track>.parquet` every 25 folds and when that track finishes. After a crash or disconnect, set `RESUME = True` and rerun: runs already saved are skipped.

In [6]:
results_df = run_evaluation(
    folds            = folds,
    tracks           = ACTIVE_TRACKS,
    targets          = TARGET_LIST,
    horizons         = HORIZON_LIST,
    models           = MODELS,
    targets_df       = targets,
    track_config     = TRACK_CONFIG,
    track_data       = track_data,
    stationary_panel = stationary_panel,
    run_by           = RUN_BY,
    results_dir      = RESULTS_DIR,  # checkpoint every 25 folds and at the end of each track
    resume           = RESUME,
)

n_err = results_df["error"].notna().sum()
print(f"Runs with errors: {n_err}")
if n_err:
    print(results_df.loc[results_df["error"].notna(), ["model", "track", "target", "error"]].drop_duplicates("error").head())

  Saved: raw/adaboost/B.parquet   new=300  total=300


  Saved: raw/adaboost/B.parquet   new=300  total=600


  Saved: raw/adaboost/B.parquet   new=300  total=900


  1000/7764 runs complete · 2m42s elapsed · ~18m18s left


  Saved: raw/adaboost/B.parquet   new=300  total=1200


...

  Saved: raw/adaboost/B.parquet   new=300  total=6900


  7000/7764 runs complete · 16m04s elapsed · ~1m45s left


  Saved: raw/adaboost/B.parquet   new=300  total=7200


  Saved: raw/adaboost/B.parquet   new=300  total=7500


  Saved: raw/adaboost/B.parquet   new=224  total=7724
Loop complete. Total rows: 7724 in 17m37s
Runs with errors: 0


## 5. Summarize

Saving merges rather than overwrites: only the rows a run recomputed are replaced, so teammates running other models or tracks are unaffected.

In [7]:
# summaries cover everything on disk for the active models/tracks, not just this run
all_results = load_results(RESULTS_DIR, models=ACTIVE_MODELS, tracks=ACTIVE_TRACKS)
summary = summarize(all_results)
summary

mse       mae  median_ae      rmse  \
model    track target   horizon                                            
adaboost B     CPIAUCSL 1        0.000009  0.002124   0.002124  0.002124   
                        3        0.000054  0.005175   0.005175  0.005175   
                        6        0.000173  0.009453   0.009453  0.009453   
               INDPRO   1        0.000094  0.005254   0.005254  0.005254   
                        3        0.000382  0.011030   0.011030  0.011030   
                        6        0.000762  0.017667   0.017667  0.017667   
               PAYEMS   1        0.000047  0.001446   0.001446  0.001446   
                        3        0.000156  0.003542   0.003542  0.003542   
                        6        0.000208  0.005959   0.005959  0.005959   
               S&P 500  1        0.001398  0.026816   0.026816  0.026816   
                        3        0.005080  0.052429   0.052429  0.052429   
                        6        0.010532  0.076761   0.076761  0.076761   

                                     mase     rmspe  
model    track target   horizon                      
adaboost B     CPIAUCSL 1        1.124561  1.611275  
                        3        1.109946  2.595573  
                        6        1.336684  1.382049  
               INDPRO   1        0.843422  2.924697  
                        3        0.921598  3.304647  
                        6        0.848843  6.800805  
               PAYEMS   1        1.131062  1.712679  
                        3        1.212382  1.352593  
                        6        1.057163  2.848880  
               S&P 500  1        0.815959  4.148862  
                        3        0.757792  1.955537  
                        6        0.763423  1.986379

In [8]:
# recession (1) vs expansion (0)
summary_disagg = disaggregate_by_usrec(load_results(RESULTS_DIR), usrec)  # all models/tracks on disk
write_csv(summary_disagg, RESULTS_DIR / "summary_disaggregated.csv")
summary_disagg

mse       mae  median_ae  \
model    track target   horizon usrec                                  
adaboost A     CPIAUCSL 1       0      0.000007  0.001878   0.001878   
                                1      0.000011  0.002792   0.002792   
                        3       0      0.000039  0.004560   0.004560   
                                1      0.000047  0.005476   0.005476   
                        6       0      0.000107  0.007519   0.007519   
...                                         ...       ...        ...   
xgboost  B     S&P 500  1       1      0.003672  0.044113   0.044113   
                        3       0      0.004278  0.048328   0.048328   
                                1      0.011069  0.080029   0.080029   
                        6       0      0.008297  0.070519   0.070519   
                                1      0.028122  0.141068   0.141068   

                                           rmse      mase     rmspe  
model    track target   horizon usrec                                
adaboost A     CPIAUCSL 1       0      0.001878  1.014579  0.475022  
                                1      0.002792  1.396021  0.691189  
                        3       0      0.004560  0.963850  0.376184  
                                1      0.005476  1.078794  0.665626  
                        6       0      0.007519  0.991175  0.512725  
...                                         ...       ...       ...  
xgboost  B     S&P 500  1       1      0.044113  1.240017  1.522456  
                        3       0      0.048328  0.700964  1.976205  
                                1      0.080029  1.117124  1.205569  
                        6       0      0.070519  0.707902  2.270912  
                                1      0.141068  1.339816  1.358906  

[192 rows x 6 columns]

In [9]:
# one row per (run, test date): y_true vs y_pred
predictions = explode_predictions(all_results)
runtime = all_results.groupby(["model", "track"])["runtime_sec"].agg(["mean", "sum"]).round(3)
display(predictions.head(), runtime)

,model,track,target,horizon,fold_id,date,y_true,y_pred
0,adaboost,B,CPIAUCSL,1,0,1972-04-01,0.002407,0.002723
1,adaboost,B,CPIAUCSL,1,1,1972-05-01,0.002401,0.001724
2,adaboost,B,CPIAUCSL,1,2,1972-06-01,0.002395,0.002341
3,adaboost,B,CPIAUCSL,1,3,1972-07-01,0.002389,0.002060
4,adaboost,B,CPIAUCSL,1,4,1972-08-01,0.004762,0.003445


,,mean,sum
model,track,,
adaboost,B,0.132,1017.557
